# 09 · RQ5: Can disagreement information improve emotion recognition?

Report section 3.6 (Figure 8, Table 10), the fusion row of Table 6, and the
stacking model of Section 2.4.

All fusion methods are learned on the validation speakers and tested once on
the test speakers. A leave-one-speaker-out check on the test set is at the end.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    P = Path("/content/drive/MyDrive/Affective Project/predictions")
else:
    P = Path("../predictions")

FILES = {
    "audio_v2_test": P / "audio_v2_test_predictions.csv",
    "audio_v2_validation": P / "audio_v2_validation_predictions.csv",
    "audio_v1_validation": P / "audio_v1_validation_predictions.csv",
    "audio_v2_history": P / "audio_v2_history.csv",
    "visual_test": P / "visual_test_predictions.csv",
    "visual_validation": P / "visual_validation_predictions.csv",
}
missing = [str(p) for p in FILES.values() if not p.exists()]
assert not missing, "Missing files:\n" + "\n".join(missing)
for k, p in FILES.items():
    print(f"{k:22s} {p}")

Mounted at /content/drive
audio_v2_test          /content/drive/MyDrive/Affective Project/predictions/audio_v2_test_predictions.csv
audio_v2_validation    /content/drive/MyDrive/Affective Project/predictions/audio_v2_validation_predictions.csv
audio_v1_validation    /content/drive/MyDrive/Affective Project/predictions/audio_v1_validation_predictions.csv
audio_v2_history       /content/drive/MyDrive/Affective Project/predictions/audio_v2_history.csv
visual_test            /content/drive/MyDrive/Affective Project/predictions/visual_test_predictions.csv
visual_validation      /content/drive/MyDrive/Affective Project/predictions/visual_validation_predictions.csv


In [2]:
D = Path("/content/drive/MyDrive/Affective Project/data") if IN_COLAB else Path("../data")
DATA = {"ratings": D / "summaryTable.csv", "actors": D / "VideoDemographics.csv"}
missing = [str(p) for p in DATA.values() if not p.exists()]
assert not missing, "Missing files:\n" + "\n".join(missing)

In [3]:
CLASSES = ["Anger", "Disgust", "Fear", "Happy", "Neutral", "Sad"]
PROB = [f"prob_{c}" for c in CLASSES]


def load_split(split):
    """One row per clip: true label and both models' predictions and probabilities.
    Clips without a visual prediction are dropped (only 1076_MTI_SAD_XX in validation)."""
    v = pd.read_csv(FILES[f"visual_{split}"])
    v = v[v["status"] == "ok"]
    a = pd.read_csv(FILES[f"audio_v2_{split}"])
    a["video_id"] = a["filename"].str.replace(".wav", "", regex=False)
    m = v[["video_id", "Speaker", "true_label", "predicted_label", *PROB]].merge(
        a[["video_id", "true_label", "predicted_label", *PROB]],
        on="video_id", suffixes=("_v", "_a"), validate="one_to_one")
    assert (m["true_label_v"] == m["true_label_a"]).all(), "label mismatch"
    m = m.rename(columns={"true_label_v": "y", "predicted_label_v": "pred_v",
                          "predicted_label_a": "pred_a"}).drop(columns="true_label_a")
    return m


def ece(p, y_idx, bins=10):
    """Expected calibration error with 10 equal-width confidence bins."""
    conf, pred = p.max(1), p.argmax(1)
    correct = (pred == y_idx).astype(float)
    edges = np.linspace(0, 1, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        s = (conf > lo) & (conf <= hi)
        if s.any():
            total += s.mean() * abs(conf[s].mean() - correct[s].mean())
    return total

In [4]:
CODE = {"A": "Anger", "D": "Disgust", "F": "Fear", "H": "Happy", "N": "Neutral", "S": "Sad"}


def add_human_ratings(df):
    s = pd.read_csv(DATA["ratings"], dtype=str).rename(columns={"FileName": "video_id"})
    df = df.merge(s[["video_id", "VoiceVote", "FaceVote", "MultiModalVote"]],
                  on="video_id", how="left", validate="one_to_one")
    assert df.VoiceVote.notna().all()
    for col, new in [("VoiceVote", "h_voice"), ("FaceVote", "h_face"), ("MultiModalVote", "h_av")]:
        df[new] = df[col].map(lambda x: frozenset(CODE[c] for c in x.split(":")))
        # a human channel is "correct" when the intended emotion is the most chosen label
        df[new + "_correct"] = [y in s for y, s in zip(df.y, df[new])]
    # humans disagree between channels when voice-only and face-only raters share no top label
    df["human_disagree"] = [len(a & b) == 0 for a, b in zip(df.h_voice, df.h_face)]
    return df

## Fusion methods

In [5]:
from scipy.optimize import minimize_scalar
from scipy.stats import binomtest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

K, EPS = len(CLASSES), 1e-12


def to_arrays(df):
    return dict(pv=df[[f"{p}_v" for p in PROB]].to_numpy(float),
                pa=df[[f"{p}_a" for p in PROB]].to_numpy(float),
                y=np.array([CLASSES.index(c) for c in df.y]),
                speaker=df.Speaker.to_numpy())


def subset(d, idx):
    return {k: v[idx] for k, v in d.items()}


def nll(p, y):
    return -np.mean(np.log(p[np.arange(len(y)), y] + EPS))


def temp_scale(p, T):
    z = np.log(p + EPS) / T
    z -= z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def fit_temperature(p, y):
    return float(minimize_scalar(lambda T: nll(temp_scale(p, T), y), bounds=(0.05, 20), method="bounded").x)


def features(pv, pa):
    """Stacking input (16 numbers per clip): 6 + 6 log-probabilities, disagreement flag,
    confidence of each model and the difference of the confidences."""
    lv, la = np.log(pv + EPS), np.log(pa + EPS)
    dis = (pv.argmax(1) != pa.argmax(1)).astype(float)[:, None]
    cv, ca = pv.max(1, keepdims=True), pa.max(1, keepdims=True)
    return np.hstack([lv, la, dis, cv, ca, cv - ca])


def fit_methods(fit):
    """Learn every fusion method on the fit data; return a predict function."""
    pv, pa, y = fit["pv"], fit["pa"], fit["y"]
    Tv, Ta = fit_temperature(pv, y), fit_temperature(pa, y)
    cv, ca = temp_scale(pv, Tv), temp_scale(pa, Ta)
    grid = np.linspace(0, 1, 21)
    w = grid[np.argmax([accuracy_score(y, (g * cv + (1 - g) * ca).argmax(1)) for g in grid])]

    def precision(p):
        pred = p.argmax(1)
        return np.array([(y[pred == k] == k).mean() if (pred == k).any() else 0 for k in range(K)])
    prec_v, prec_a = precision(cv), precision(ca)

    clf = LogisticRegression(C=1.0, max_iter=2000).fit(features(cv, ca), y)
    params = dict(T_visual=Tv, T_audio=Ta, w_visual=float(w))

    def predict(ev):
        ev_cv, ev_ca = temp_scale(ev["pv"], Tv), temp_scale(ev["pa"], Ta)
        yv, ya = ev_cv.argmax(1), ev_ca.argmax(1)
        out = {
            "Visual only": yv,
            "Audio only": ya,
            "Average (raw)": (ev["pv"] + ev["pa"]).argmax(1),
            "Product": (np.log(ev["pv"] + EPS) + np.log(ev["pa"] + EPS)).argmax(1),
            "Weighted average": (w * ev_cv + (1 - w) * ev_ca).argmax(1),
            "Most confident": np.where(ev["pv"].max(1) >= ev["pa"].max(1), ev["pv"].argmax(1), ev["pa"].argmax(1)),
            "Most confident (cal.)": np.where(ev_cv.max(1) >= ev_ca.max(1), yv, ya),
            "Per-class trust rule": np.where(yv != ya, np.where(prec_v[yv] >= prec_a[ya], yv, ya), yv),
            "Stacking": clf.predict(features(ev_cv, ev_ca)),
            "Upper bound": np.where(yv == ev["y"], yv, ya),
        }
        return out, ev_cv, ev_ca
    return predict, params

## Statistics: speaker bootstrap and McNemar test

In [6]:
def speaker_bootstrap(y, a, b, speakers, n=2000, seed=0):
    """95% CI of the accuracy difference a - b, resampling whole speakers."""
    rng = np.random.default_rng(seed)
    spk = np.unique(speakers)
    idx_by = {s: np.where(speakers == s)[0] for s in spk}
    diffs = []
    for _ in range(n):
        idx = np.concatenate([idx_by[s] for s in rng.choice(spk, len(spk))])
        diffs.append((a[idx] == y[idx]).mean() - (b[idx] == y[idx]).mean())
    return np.percentile(diffs, [2.5, 97.5])


def mcnemar(y, a, b):
    ok_a, ok_b = a == y, b == y
    n10, n01 = int((ok_a & ~ok_b).sum()), int((~ok_a & ok_b).sum())
    return n10, n01, binomtest(n10, n10 + n01, 0.5).pvalue

## Step 1: Are the confidence values reliable? (Figure 8a)

In [7]:
val, test = to_arrays(load_split("validation")), to_arrays(load_split("test"))
predict, params = fit_methods(val)
preds, cal_v, cal_a = predict(test)
print("Fitted on the validation set:", {k: round(v, 2) for k, v in params.items()})

y = test["y"]
rows = []
for model, raw, cal in [("Audio", test["pa"], cal_a), ("Visual", test["pv"], cal_v)]:
    for stage, p in [("raw", raw), ("temperature-scaled", cal)]:
        rows.append({"model": model, "stage": stage, "average_confidence_%": 100 * p.max(1).mean(),
                     "accuracy_%": 100 * (p.argmax(1) == y).mean(), "ECE": ece(p, y)})
pd.DataFrame(rows).round(3)

Fitted on the validation set: {'T_visual': 2.14, 'T_audio': 1.12, 'w_visual': 0.6}


,model,stage,average_confidence_%,accuracy_%,ECE
0,Audio,raw,68.206,67.784,0.022
1,Audio,temperature-scaled,64.626,67.784,0.033
2,Visual,raw,71.233,68.041,0.037
3,Visual,temperature-scaled,54.393,68.041,0.145


## Step 2: Combining the two models (Figure 8b, Table 10)

In [8]:
ref = preds["Visual only"]
dis = preds["Visual only"] != preds["Audio only"]
rows = []
for name, p in preds.items():
    row = {"method": name, "accuracy_%": 100 * accuracy_score(y, p),
           "macro_F1_%": 100 * f1_score(y, p, average="macro"),
           "acc_when_models_agree_%": 100 * accuracy_score(y[~dis], p[~dis]),
           "acc_when_models_disagree_%": 100 * accuracy_score(y[dis], p[dis])}
    if name not in ("Visual only", "Upper bound"):
        lo, hi = speaker_bootstrap(y, p, ref, test["speaker"])
        n10, n01, pval = mcnemar(y, p, ref)
        row.update({"gain_vs_visual": 100 * (accuracy_score(y, p) - accuracy_score(y, ref)),
                    "CI_low": 100 * lo, "CI_high": 100 * hi, "p_McNemar": pval,
                    "fixed_vs_visual": n10, "broken_vs_visual": n01})
    rows.append(row)
table10 = pd.DataFrame(rows).set_index("method")
table10.loc["Upper bound", "gain_vs_visual"] = table10.loc["Upper bound", "accuracy_%"] - table10.loc["Visual only", "accuracy_%"]
table10.round(3)

,accuracy_%,macro_F1_%,acc_when_models_agree_%,acc_when_models_disagree_%,gain_vs_visual,CI_low,CI_high,p_McNemar,fixed_vs_visual,broken_vs_visual
method,,,,,,,,,,
Visual only,68.041,67.331,94.716,42.642,NaN,NaN,NaN,NaN,NaN,NaN
Audio only,67.784,67.680,94.716,42.138,-0.258,-6.509,5.477,0.908,335.0,339.0
Average (raw),80.348,80.094,94.716,66.667,12.307,8.858,15.470,0.000,259.0,68.0
Product,80.735,80.531,94.716,67.421,12.693,9.600,15.663,0.000,263.0,66.0
Weighted average,80.026,79.832,94.716,66.038,11.985,8.247,15.593,0.000,269.0,83.0
Most confident,77.835,77.455,94.716,61.761,9.794,6.186,13.338,0.000,230.0,78.0
Most confident (cal.),75.838,75.712,94.716,57.862,7.796,3.350,12.178,0.000,275.0,154.0
Per-class trust rule,73.131,71.303,94.716,52.579,5.090,2.181,8.083,0.000,198.0,119.0
Stacking,81.830,81.477,93.395,70.818,13.789,9.434,17.854,0.000,299.0,85.0


In [9]:
s = table10.loc["Stacking"]
print(f"Stacking: accuracy {s['accuracy_%']:.1f}%, macro-F1 {s['macro_F1_%'] / 100:.3f}, "
      f"gain {s['gain_vs_visual']:+.1f} points (95% CI {s['CI_low']:+.1f} to {s['CI_high']:+.1f}), "
      f"McNemar p = {s['p_McNemar']:.1e}")
print(f"Compared with the visual model: {int(s['fixed_vs_visual'])} errors corrected, "
      f"{int(s['broken_vs_visual'])} new errors")
print(f"Clips where the models disagree: {dis.sum()}; visual alone right "
      f"{table10.loc['Visual only', 'acc_when_models_disagree_%']:.1f}%, stacking right "
      f"{s['acc_when_models_disagree_%']:.1f}%")

Stacking: accuracy 81.8%, macro-F1 0.815, gain +13.8 points (95% CI +9.4 to +17.9), McNemar p = 5.0e-29
Compared with the visual model: 299 errors corrected, 85 new errors
Clips where the models disagree: 795; visual alone right 42.6%, stacking right 70.8%


## Fusion compared with human raters who saw and heard the clips (Table 6, last row)

McNemar test on the clips where exactly one of the two (stacking or humans) is correct.

In [10]:
h = add_human_ratings(load_split("test"))
stack_ok = np.array(CLASSES)[preds["Stacking"]] == h.y.to_numpy()
human_ok = h.h_av_correct.to_numpy()
n10, n01 = int((stack_ok & ~human_ok).sum()), int((~stack_ok & human_ok).sum())
print(f"Stacking {100 * stack_ok.mean():.1f}% vs. humans (audio and video) {100 * human_ok.mean():.1f}%")
print(f"Only stacking correct: {n10}, only humans correct: {n01}, "
      f"McNemar p = {binomtest(n10, n10 + n01, 0.5).pvalue:.1e}")

Stacking 81.8% vs. humans (audio and video) 76.5%
Only stacking correct: 216, only humans correct: 133, McNemar p = 1.0e-05


## Robustness check: leave-one-speaker-out on the test set

For each test speaker, all methods are learned on the other 18 test speakers and
applied to the held-out speaker.

In [11]:
loso = {k: np.empty(len(y), dtype=int) for k in preds}
for s in np.unique(test["speaker"]):
    tr, te = np.where(test["speaker"] != s)[0], np.where(test["speaker"] == s)[0]
    pred_s, _ = fit_methods(subset(test, tr))[0](subset(test, te))[:2]
    for k in pred_s:
        loso[k][te] = pred_s[k]
pd.Series({k: 100 * accuracy_score(y, p) for k, p in loso.items()}, name="LOSO accuracy %").round(1)

,LOSO accuracy %
Visual only,68.0
Audio only,67.8
Average (raw),80.3
Product,80.7
Weighted average,79.4
Most confident,77.8
Most confident (cal.),75.8
Per-class trust rule,73.6
Stacking,82.0
Upper bound,89.6
